# Journal-ready contract price, volume, utility, and PPA-choice evolution under mutation

This notebook replaces the earlier compact diagnostic dashboards with a **journal-oriented figure and table workflow**. It reads:

```text
Output files (Risk Neutral, Mutation, Verified)/Simulation_Plot_Data_All_Matches.csv
```

It is designed to be saved and run inside:

```text
Code_Submission/simulation_mutation/
```

The notebook writes new outputs under:

```text
Output files (Risk Neutral, Mutation, Verified)/Contract_Price_Volume_Utility_Changes/journal_figures/
Output files (Risk Neutral, Mutation, Verified)/Contract_Price_Volume_Utility_Changes/journal_tables/
```

## Main changes in this version

1. **Paired baseline-to-mutation long table** by `match_id` and mutation scenario.
2. **Profile-share response figure** by mutation family and target shift.
3. **Family-specific transition matrices** for the most severe target in each family.
4. **Contract-term change figure** separating:
   - strike-price change for active PPA → active PPA observations;
   - fixed-volume change only for `Fix → Fix` observations;
   - mean delivered-volume proxy change for cross-profile comparison.
5. **Seller metric vs buyer participation slack movement** figure.
6. **Correlation / diagnostic spillover heatmap** when correlation-style columns exist in the input CSV.
7. **Reusable numerical tables** for later analysis, captions, and manuscript checks.

## Volume interpretation

Raw `volume_mw` is only directly comparable for `Fix → Fix`. Cross-profile comparisons use the mean delivered-volume proxy:

```text
Fix         -> optimized fixed volume_mw
AsG         -> scenario mean seller generation
AsC         -> scenario mean buyer demand
No Contract -> 0
```

This proxy is saved as `mean_delivered_volume_proxy_mw`. It should be described as a proxy, not as a universal contract-volume variable.


In [ ]:
# ============================================================
# USER CONTROLS
# ============================================================

from pathlib import Path
import textwrap


# Leave as None when this notebook is saved/run inside Code_Submission/simulation_mutation.
# Set to an absolute path only if Jupyter cannot resolve the folder correctly.
simulation_mutation_dir_override = None

# Leave as None to use the default aggregate simulation CSV below.
# Set to an absolute path if you want to use a different Simulation_Plot_Data_All_Matches.csv.
plot_data_csv_override = None

output_folder_name = "Output files (Risk Neutral, Mutation, Verified)"
plot_data_filename = "Simulation_Plot_Data_All_Matches.csv"
output_subdir = "Contract_Price_Volume_Utility_Changes"
journal_figure_subdir = "journal_figures"
journal_table_subdir = "journal_tables"

# Optional filters. Leave as None to keep all simulated mutation scenarios.
filter_match_ids = None                 # e.g. [1, 2, 3]
filter_mutation_families = None         # e.g. ["shape", "basis"] or paper-style labels
filter_target_pair_labels = None        # e.g. ["shape_corr"]
filter_scenario_names = None            # e.g. ["Mutation__..."]
filter_target_shifts = None             # e.g. [-0.1, -0.3, -0.5]

# Column-selection controls. The code uses the first existing numeric column in each list.
strike_price_column_candidates = [
    "strike_price_mwh", "selected_strike_price_mwh", "strike_price", "price_mwh", "contract_price_mwh", "pi"
]
fixed_volume_column_candidates = [
    "volume_mw", "selected_volume_mw", "fixed_volume_mw", "contract_volume_mw", "q_mw", "q"
]
generation_mean_column_candidates = [
    "generation_simulated_mean", "generation_mean_ref", "generation_original_mean", "generation_mean",
    "seller_volume_mean_used", "mean_generation_mw", "mean_g_mw", "g_mean"
]
demand_mean_column_candidates = [
    "demand_simulated_mean", "demand_mean_ref", "demand_original_mean", "demand_mean",
    "buyer_volume_mean_used", "mean_demand_mw", "mean_d_mw", "d_mean"
]
seller_metric_column_candidates = [
    "seller_risk_adjusted_objective", "seller_objective", "seller_objective_value",
    "seller_risk_adjusted_exposure", "seller_exposure_risk_adjusted", "seller_utility"
]
buyer_metric_column_candidates = [
    "buyer_risk_adjusted_exposure", "buyer_objective", "buyer_objective_value", "buyer_utility"
]
buyer_slack_column_candidates = [
    "buyer_participation_slack", "participation_slack", "buyer_slack", "buyer_feasibility_slack"
]

# Seller-metric interpretation for the seller-vs-buyer movement figure and quadrant table.
# Use None to infer from the selected seller metric column name.
# For exposure/objective columns, lower is usually better; for utility columns, higher is usually better.
seller_metric_lower_is_better = None

# Diagnostic delta columns are built for numeric columns whose names contain these patterns.
# These are used for the optional correlation/spillover heatmap.
diagnostic_column_patterns = ["corr", "correlation", "volume_mismatch", "price_spread", "relation_index"]
diagnostic_exclude_patterns = ["target", "desired", "mutation_level", "scenario_order", "plot_order"]
correlation_heatmap_patterns = ["corr", "correlation"]

# Output controls.
make_plots = True
save_png = True
save_pdf = False
save_svg = False
display_plots_in_notebook = True
show_figure_reading_notes = True
show_preview_tables = True

# Journal figure switches.
make_profile_share_figure = True
make_transition_matrix_figure = True
make_contract_terms_figure = True
make_objective_slack_figure = True
make_match_switch_heatmap = True
make_correlation_spillover_heatmap = True

# Plot-size controls.
figure_dpi = 220
max_matches_in_switch_heatmap = 40
max_scenarios_in_switch_heatmap = 16
max_top_movers_per_metric = 100

# Ordering and display labels.
contract_type_order = ["Fix", "AsC", "AsG", "No Contract", "Unknown"]
preferred_family_order = [
    "Generation-load mismatch",
    "Seller-buyer nodal price decoupling",
    "Buyer load-price intensification",
    "Seller generation-price cannibalization",
]

# Metrics saved in numerical summaries.
metrics_to_summarize = [
    "delta_strike_price_mwh",
    "delta_fixed_volume_mw",
    "delta_mean_delivered_volume_proxy_mw",
    "delta_seller_metric",
    "delta_buyer_metric",
    "delta_total_metric",
    "delta_buyer_participation_slack",
]

# Figure metric display names. Values not present in the data are skipped automatically.
metric_display_names = {
    "delta_strike_price_mwh": "Δ Strike price ($/MWh)",
    "delta_fixed_volume_mw": "Δ Fixed volume (MW; Fix→Fix only)",
    "delta_mean_delivered_volume_proxy_mw": "Δ Mean delivered-volume proxy (MW)",
    "delta_seller_metric": "Δ Seller metric",
    "delta_buyer_metric": "Δ Buyer metric",
    "delta_total_metric": "Δ Total metric",
    "delta_buyer_participation_slack": "Δ Buyer participation slack",
}

# ============================================================
# FIGURE-SPECIFIC STYLE CONTROLS
# ============================================================
# Each block is independent. Set show_suptitle=True only when you want a major figure title.
# hspace controls the vertical gap between subplot rows; wspace controls the horizontal gap.

profile_share_figure_style = {
    "ncols": 2,
    "fig_width": 14.0,
    "row_height": 4.8,
    "min_height": 5.2,
    "hspace": 0.28,
    "wspace": 0.24,
    "left": None,
    "right": None,
    "bottom": None,
    "top": 0.94,
    "show_suptitle": False,
    "suptitle": "Selected PPA profile shares by mutation family",
    "suptitle_fontsize": 14,
    "suptitle_y": 1.02,
    "panel_title_fontsize": 14,
    "panel_title_pad": 8,
    "axis_label_fontsize": 14,
    "x_label_pad": 5,
    "y_label_pad": 5,
    "tick_label_fontsize": 14,
    "x_tick_rotation": 0,
    "line_width": 1.8,
    "marker_size": 5,
    "show_legend": True,
    "legend_loc": "best",
    "legend_fontsize": 14,
    "legend_frameon": True,
    "x_axis_label": "Correlation shift",
    "y_axis_label": "Selected PPA structure share",
}

transition_matrix_figure_style = {
    "ncols": 2,
    "fig_width": 13.5,
    "row_height": 5.1,
    "min_height": 5.5,
    "hspace": 0.24,
    "wspace": 0.24,
    "bottom": 0.09,
    "top": 0.94,
    "show_suptitle": False,
    "suptitle": "Baseline-to-mutation transition matrices at the most severe target in each family",
    "suptitle_fontsize": 14,
    "suptitle_y": 1.02,
    "panel_title_fontsize": 14,
    "panel_title_pad": 8,
    "axis_label_fontsize": 14,
    "tick_label_fontsize": 14,
    "cell_annotation_fontsize": 14,
    "cell_annotation_fontweight": "normal",
    "cell_annotation_color_dark": "black",
    "cell_annotation_color_light": "white",
    "text_color_threshold": 0.55,
    # Leave per-panel axis titles blank; the shared note below identifies row/column meanings.
    "x_axis_label": "",
    "y_axis_label": "",
    "show_shared_axis_note": False,
    "shared_axis_note": "Rows = baseline result; columns = dynamic mutation result.",
    "shared_axis_note_fontsize": 9,
    "shared_axis_note_y": 0.025,
    # Heatmap color is now row percentage, not count, so all panels use the same 0–100% scale.
    "cmap": "Blues",
    # "colorbar_label": "row share",
    # "colorbar_label_fontsize": 14,
    "colorbar_tick_fontsize": 14,
    "colorbar_fraction": 0.035,
    "colorbar_pad": 0.02,
    "colorbar_shrink": 0.90,
}

contract_terms_figure_style = {
    "fig_width_per_column": 4.4,
    "fig_height_per_row": 4.1,
    "min_width": 8.0,
    "min_height": 6.0,
    "hspace": 0.18,
    "wspace": 0.30,
    "bottom": 0.08,
    "top": 0.94,
    "show_suptitle": False,
    "suptitle": "Contract-term changes under mutation\nMedian with IQR and 10–90% ranges across matches",
    "suptitle_fontsize": 14,
    "suptitle_y": 1.02,
    "show_column_titles": True,
    "panel_title_fontsize": 14,
    "panel_title_pad": 8,
    "wrap_column_titles": True,
    # Measure titles against the rendered width of their own subfigure.
    "column_title_width_fraction": 0.99,
    "column_title_extra_height_per_line": 0.00,
    "axis_label_fontsize": 16,
    "x_label_pad": 7,
    "y_label_pad": 5,
    "tick_label_fontsize": 14,
    "x_tick_rotation": 0,
    "show_n_labels": True,
    # "xtick_second_line" puts n below the shift tick label and removes the prior overlap.
    # Alternatives: "inside_bottom", "below_axis", or "none".
    "n_label_mode": "xtick_second_line",
    "n_label_fontsize": 7,
    "n_label_rotation": 0,
    "n_label_y_axes": -0.20,
    "n_label_inside_offset": 0.04,
    "x_axis_label": "Correlation shift",
    "metric_y_labels": {
        "delta_strike_price_mwh": "Δ strike price\n($/MWh)",
        "delta_fixed_volume_mw": "Δ fixed volume\n(MW; Fix→Fix)",
        "delta_mean_delivered_volume_proxy_mw": "Δ Mean delivered\nvolume (MW)",
    },
    "wrap_y_axis_labels": True,
    "y_axis_label_max_chars": 22,
    "interval_line_width_10_90": 1.1,
    "interval_line_width_iqr": 4.0,
    "median_line_width": 1.5,
    "median_marker_size": 5,
    "show_legend": True,
    "legend_loc": "best",
    "legend_fontsize": 14,
    "legend_frameon": True,
}

objective_slack_figure_style = {
    "ncols": 2,
    "fig_width": 13.5,
    "row_height": 4.9,
    "min_height": 5.5,
    "hspace": 0.35,
    "wspace": 0.34,
    "bottom": 0.08,
    "right": 0.84,
    "top": 0.94,
    "show_suptitle": False,
    "suptitle": (
        "Seller metric and buyer participation-slack movement by mutation family\n"
        "Small points are match-scenario observations; diamonds are target-level medians."
    ),
    "suptitle_fontsize": 14,
    "suptitle_y": 1.02,
    "panel_title_fontsize": 14,
    "panel_title_pad": 8,
    "axis_label_fontsize": 14,
    "x_label_pad": 5,
    "y_label_pad": 5,
    "tick_label_fontsize": 14,
    "scatter_size": 14,
    "scatter_alpha": 0.35,
    "centroid_size": 70,
    "show_centroid_labels": False,
    "centroid_label_fontsize": 14,
    "x_axis_label": "Δ Seller exposure index",
    "y_axis_label": "Δ Buyer slack",
    "cmap": "viridis",
    "colorbar_label": "|correlation shift|",
    "colorbar_label_fontsize": 14,
    "colorbar_tick_fontsize": 14,
    # Fixed figure coordinates keep the colorbar outside the panels when font sizes increase.
    "colorbar_position": [0.87, 0.18, 0.018, 0.66],
    "colorbar_fraction": 0.030,
    "colorbar_pad": 0.02,
    "colorbar_shrink": 0.85,
}

match_switch_heatmap_figure_style = {
    "min_width": 8.0,
    "scenario_width": 0.7,
    "min_height": 6.0,
    "match_height": 0.22,
    "height_padding": 2.8,
    "show_title": True,
    "title": "Selected profile by match and mutation scenario\n(matches ranked by switch activity)",
    "title_fontsize": 14,
    "title_pad": 8,
    "axis_label_fontsize": 10,
    "tick_label_fontsize": 8,
    "x_tick_rotation": 45,
    "y_tick_label_fontsize": 7,
    "colorbar_label_fontsize": 9,
    "colorbar_tick_fontsize": 8,
}

correlation_spillover_heatmap_figure_style = {
    "min_width": 10.0,
    "column_width": 0.7,
    "min_height": 6.0,
    "row_height": 0.35,
    "height_padding": 2.5,
    "show_title": True,
    "title": "Median change in correlation diagnostics by mutation scenario",
    "title_fontsize": 14,
    "title_pad": 8,
    "tick_label_fontsize": 9,
    "x_tick_rotation": 45,
    "cell_annotation_fontsize": 7,
    "colorbar_label": "median baseline-to-mutation delta",
    "colorbar_label_fontsize": 9,
    "colorbar_tick_fontsize": 8,
}

# Short printed notes shown next to each saved/displayed figure.
figure_reading_notes = {
    "fig_profile_share_by_family": (
        "Each panel is one mutation family. The x-axis is the target correlation shift; "
        "the y-axis is the share of matches selecting each PPA profile. Lines show how the selected-profile mix changes as the mutation intensifies."
    ),
    "fig_transition_matrices_by_family_severe": (
        "Each panel uses the most severe target shift in that mutation family. Rows are baseline selected profiles; columns are dynamic mutation selected profiles. "
        "Each cell reports count and row percentage. The color scale is the row percentage, so darker cells indicate a larger share of a baseline row moving to that dynamic profile."
    ),
    "fig_contract_terms_by_family": (
        "Columns are mutation families and rows are contract-term outcomes. For each target shift, the dot is the median baseline-to-mutation change across matches; "
        "the thick vertical segment is the IQR and the thin segment is the 10–90% range. The second x-tick line reports the valid sample size."
    ),
    "fig_objective_slack_movement": (
        "Each dot is one match-scenario observation. The x-axis is the baseline-to-mutation change in the selected seller metric, and the y-axis is the change in buyer participation slack. "
        "Zero lines separate no change from positive/negative changes. Diamonds are target-level medians; color shows absolute correlation-shift severity."
    ),
    "fig_match_switch_heatmap": (
        "Rows are matches ranked by switching activity; columns are mutation scenarios. Each cell shows the selected PPA profile for that match-scenario. "
        "Use this as a diagnostic for which matches are most sensitive to mutation rather than as an aggregate effect-size figure."
    ),
    "fig_correlation_spillover_heatmap": (
        "Rows are mutation-family/target-shift scenarios and columns are correlation-style diagnostics. Cell values are median baseline-to-mutation diagnostic changes. "
        "The diverging color scale centers on zero, so sign indicates direction and intensity indicates magnitude."
    ),
}


In [ ]:
contract_terms_figure_style = {
    "fig_width_per_column": 4.4,
    "fig_height_per_row": 4.1,
    "min_width": 8.0,
    "min_height": 6.0,
    "hspace": 0.18,
    "wspace": 0.30,
    "bottom": 0.08,
    "top": 0.92,
    "show_suptitle": False,
    "suptitle": "Contract-term changes under mutation\nMedian with IQR and 10–90% ranges across matches",
    "suptitle_fontsize": 14,
    "suptitle_y": 1.02,
    "show_column_titles": True,
    "panel_title_fontsize": 14,
    "panel_title_pad": 8,
    "wrap_column_titles": True,
    # Measure titles against the rendered width of their own subfigure.
    "column_title_width_fraction": 0.99,
    "column_title_extra_height_per_line": 0.00,
    "axis_label_fontsize": 14,
    "x_label_pad": 7,
    "y_label_pad": 5,
    "tick_label_fontsize": 14,
    "x_tick_rotation": 0,
    "show_n_labels": True,
    # "xtick_second_line" puts n below the shift tick label and removes the prior overlap.
    # Alternatives: "inside_bottom", "below_axis", or "none".
    "n_label_mode": "xtick_second_line",
    "n_label_fontsize": 6,
    "n_label_rotation": 0,
    "n_label_y_axes": -0.20,
    "n_label_inside_offset": 0.04,
    "x_axis_label": "Correlation shift",
    "metric_y_labels": {
        "delta_strike_price_mwh": "Δ Strike price\n($/MWh)",
        "delta_fixed_volume_mw": "Δ Fixed volume\n(MW; Fix→Fix)",
        "delta_mean_delivered_volume_proxy_mw": "Δ M ean delivered\nvolume (MW)",
    },
    "wrap_y_axis_labels": True,
    "y_axis_label_max_chars": 22,
    "interval_line_width_10_90": 1.1,
    "interval_line_width_iqr": 4.0,
    "median_line_width": 1.5,
    "median_marker_size": 5,
    "show_legend": True,
    "legend_loc": "best",
    "legend_fontsize": 14,
    "legend_frameon": True,
}

In [ ]:
# ============================================================
# Submission dtype helpers
# ============================================================
# These wrappers reduce DataFrame memory use without changing the financial
# calculations: identifiers/counters are downcast, repeated labels become
# categoricals, and continuous numerical columns remain float64.

import math
import re
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.ticker import PercentFormatter
from IPython.display import Image, display

NOTEBOOK_CWD = Path.cwd().resolve()

plt.rcParams.update({
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def _as_path_or_none(value) -> Optional[Path]:
    if value is None:
        return None
    text = str(value).strip()
    return Path(text).expanduser() if text else None

def _dedupe_paths(paths) -> list[Path]:
    out, seen = [], set()
    for path in paths:
        if path is None:
            continue
        p = Path(path).expanduser()
        try:
            key = str(p.resolve())
        except Exception:
            key = str(p)
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out

def candidate_simulation_mutation_dirs(max_parent_depth: int = 6) -> list[Path]:
    override = _as_path_or_none(simulation_mutation_dir_override)
    anchors = ([override] if override is not None else []) + [NOTEBOOK_CWD] + list(NOTEBOOK_CWD.parents)[:max_parent_depth]
    dirs = []
    for anchor in anchors:
        p = Path(anchor).expanduser()
        if p.name == "simulation_mutation":
            dirs.append(p)
        if (p / output_folder_name).exists():
            dirs.append(p)
        dirs.append(p / "simulation_mutation")
    return [p.resolve() if p.exists() else p for p in _dedupe_paths(dirs)]

def resolve_input_csv() -> tuple[Path, Path]:
    override = _as_path_or_none(plot_data_csv_override)
    sim_dir_override = _as_path_or_none(simulation_mutation_dir_override)
    if override is not None:
        if not override.exists():
            raise FileNotFoundError(f"plot_data_csv_override not found: {override}")
        sim_dir = sim_dir_override or (override.parent.parent if override.parent.name == output_folder_name else override.parent)
        return sim_dir.resolve(), override.resolve()
    candidates = [sim_dir / output_folder_name / plot_data_filename for sim_dir in candidate_simulation_mutation_dirs()]
    existing = [p for p in _dedupe_paths(candidates) if p.exists()]
    if not existing:
        raise FileNotFoundError("Could not locate Simulation_Plot_Data_All_Matches.csv. Set plot_data_csv_override if needed.")
    input_path = existing[0].resolve()
    return input_path.parent.parent.resolve(), input_path

simulation_mutation_dir, input_csv = resolve_input_csv()
out_dir = (simulation_mutation_dir / output_folder_name / output_subdir).resolve()
figure_dir = out_dir / journal_figure_subdir
table_dir = out_dir / journal_table_subdir
figure_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

_PD_READ_CSV = pd.read_csv
_PD_READ_EXCEL = pd.read_excel

_INTEGER_DTYPE_CANDIDATES = {
    "match_id": np.int32,
    "hour": np.int16,
    "hour_index": np.int16,
    "replication": np.int16,
    "case_order": np.int16,
    "enabled": np.int8,
    "rank": np.int32,
}

_CATEGORY_DTYPE_CANDIDATES = {
    "case_id",
    "case_family",
    "case_label",
    "combined_category",
    "metric",
    "mutation_axis",
    "mutation_direction",
    "mutation_family",
    "mutation_label",
    "ppa_type",
    "profile_type",
    "risk_group",
    "risk_label",
    "scenario_name",
    "scenario_type",
    "solution_type",
    "status",
    "variable",
    "var_i",
    "var_j",
}


def _integer_dtype_fits(values, dtype) -> bool:
    if len(values) == 0:
        return True
    info = np.iinfo(dtype)
    return float(np.nanmin(values)) >= info.min and float(np.nanmax(values)) <= info.max


def optimize_dataframe_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Conservatively compact non-financial columns after file loading."""
    if not isinstance(df, pd.DataFrame) or df.empty:
        return df

    for col in df.columns:
        series = df[col]
        if pd.api.types.is_integer_dtype(series.dtype):
            df[col] = pd.to_numeric(series, downcast="integer")

    for col, dtype in _INTEGER_DTYPE_CANDIDATES.items():
        if col not in df.columns:
            continue
        numeric = pd.to_numeric(df[col], errors="coerce")
        if numeric.isna().any():
            continue
        values = numeric.to_numpy(dtype="float64", copy=False)
        rounded = np.rint(values)
        if np.array_equal(values, rounded) and _integer_dtype_fits(rounded, dtype):
            df[col] = rounded.astype(dtype, copy=False)

    n_rows = len(df)
    for col in _CATEGORY_DTYPE_CANDIDATES.intersection(df.columns):
        series = df[col]
        if pd.api.types.is_categorical_dtype(series.dtype):
            continue
        if not (pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype)):
            continue
        non_null = series.dropna()
        if non_null.empty:
            continue
        n_unique = int(non_null.nunique())
        if n_unique <= min(128, max(2, n_rows // 2)):
            df[col] = series.astype("category")

    return df


def read_csv_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_CSV(*args, **kwargs))


def read_excel_optimized(*args, **kwargs) -> pd.DataFrame:
    return optimize_dataframe_dtypes(_PD_READ_EXCEL(*args, **kwargs))


In [ ]:
# ============================================================
# DATA STANDARDIZATION AND PAIRED COMPARISON LOGIC
# ============================================================


def _numeric_series(df: pd.DataFrame, col: str) -> pd.Series:
    if col not in df.columns:
        return pd.Series(np.nan, index=df.index, dtype="float64")
    return pd.to_numeric(df[col], errors="coerce")


def _first_existing_column(df: pd.DataFrame, candidates: list[str], require_numeric: bool = False) -> Optional[str]:
    for col in candidates:
        if col in df.columns:
            if not require_numeric:
                return col
            s = pd.to_numeric(df[col], errors="coerce")
            if s.notna().any():
                return col
    return None


def _choose_numeric_series(df: pd.DataFrame, candidates: list[str]) -> tuple[pd.Series, Optional[str]]:
    col = _first_existing_column(df, candidates, require_numeric=True)
    if col is None:
        return pd.Series(np.nan, index=df.index, dtype="float64"), None
    return pd.to_numeric(df[col], errors="coerce"), col


def _short_text(value, max_len: int = 28) -> str:
    text = str(value)
    return text[: max_len - 3] + "..." if len(text) > max_len else text


def _format_shift(value) -> str:
    if pd.isna(value):
        return "NA"
    v = float(value)
    if abs(v) < 5e-12:
        return "Baseline"
    return f"{v:+.2f}"


def _normalize_contract_type_value(value) -> str:
    text = str(value).strip()
    low = text.lower()
    aliases = {
        "fixed": "Fix",
        "fixed-volume": "Fix",
        "fixed volume": "Fix",
        "fix": "Fix",
        "asg": "AsG",
        "as-generated": "AsG",
        "as generated": "AsG",
        "asgen": "AsG",
        "asc": "AsC",
        "as-consumed": "AsC",
        "as consumed": "AsC",
        "ascon": "AsC",
        "no contract": "No Contract",
        "nocontract": "No Contract",
        "none": "No Contract",
        "nan": "Unknown",
        "": "Unknown",
    }
    return aliases.get(low, text if text else "Unknown")


def selected_contract_type(df: pd.DataFrame) -> pd.Series:
    if "selected_profile_for_switch" in df.columns:
        raw = df["selected_profile_for_switch"].fillna("").astype(str).str.strip()
    elif "profile_type" in df.columns:
        raw = df["profile_type"].fillna("").astype(str).str.strip()
    elif "selected_profile" in df.columns:
        raw = df["selected_profile"].fillna("").astype(str).str.strip()
    else:
        raw = pd.Series("Unknown", index=df.index)

    ppa = df.get("ppa_type", pd.Series("", index=df.index)).fillna("").astype(str).str.strip()
    no_contract = ppa.str.lower().eq("no contract") | raw.str.lower().isin(
        ["no contract", "n/a", "nan", "none", ""]
    )
    raw = raw.mask(no_contract, "No Contract")
    return raw.map(_normalize_contract_type_value)


def normalize_family_value(value, scenario_name: str = "") -> str:
    text = str(value if pd.notna(value) else "").strip()
    probe = f"{text} {scenario_name}".lower().replace("_", " ").replace("-", " ")

    if "shape" in probe or "profile" in probe or "generation demand" in probe:
        return "Generation-load mismatch"
    if "basis" in probe or ("seller" in probe and "buyer" in probe and "price" in probe):
        return "Seller-buyer nodal price decoupling"
    if "load" in probe or "demand price" in probe or "buyer price" in probe:
        return "Buyer load-price intensification"
    if "cannibal" in probe or "generation price" in probe or "seller side" in probe:
        return "Seller generation-price cannibalization"
    if text and text.lower() not in ["nan", "none"]:
        return text
    return "Mutation scenario"


def infer_signed_target_shift(row: pd.Series) -> float:
    for col in ["target_physical_shift", "target_delta", "physical_target_shift", "delta_rho_phys"]:
        if col in row.index:
            value = pd.to_numeric(row[col], errors="coerce")
            if pd.notna(value):
                return float(value)

    for col in ["desired_physical_shift_abs", "mutation_level_abs", "target_abs", "target_shift_abs"]:
        if col in row.index:
            value = pd.to_numeric(row[col], errors="coerce")
            if pd.notna(value):
                fam = normalize_family_value(row.get("mutation_family", ""), row.get("scenario_name", ""))
                sign = 1.0 if fam == "Buyer load-price intensification" else -1.0
                return sign * abs(float(value))

    if "scenario_order" in row.index:
        value = pd.to_numeric(row["scenario_order"], errors="coerce")
        if pd.notna(value):
            return float(value)
    return np.nan


def find_diagnostic_columns(df: pd.DataFrame) -> list[str]:
    cols = []
    for col in df.columns:
        low = col.lower()
        if any(p in low for p in diagnostic_column_patterns) and not any(p in low for p in diagnostic_exclude_patterns):
            s = pd.to_numeric(df[col], errors="coerce")
            if s.notna().any():
                cols.append(col)
    return sorted(set(cols))


def standardize_result_columns(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    out = df.copy()

    out["match_id"] = pd.to_numeric(out["match_id"], errors="coerce").astype("Int64")
    out = out.dropna(subset=["match_id"]).copy()
    out["match_id"] = out["match_id"].astype(int)

    out["selected_contract_type"] = selected_contract_type(out)

    out["strike_price_mwh_std"], strike_source = _choose_numeric_series(out, strike_price_column_candidates)
    out["fixed_volume_mw_std"], fixed_volume_source = _choose_numeric_series(out, fixed_volume_column_candidates)
    out["generation_mean_mw"], generation_mean_source = _choose_numeric_series(out, generation_mean_column_candidates)
    out["demand_mean_mw"], demand_mean_source = _choose_numeric_series(out, demand_mean_column_candidates)
    out["seller_metric_value"], seller_metric_source = _choose_numeric_series(out, seller_metric_column_candidates)
    out["buyer_metric_value"], buyer_metric_source = _choose_numeric_series(out, buyer_metric_column_candidates)
    out["buyer_participation_slack_value"], buyer_slack_source = _choose_numeric_series(out, buyer_slack_column_candidates)

    out["total_metric_value"] = out["seller_metric_value"] + out["buyer_metric_value"]

    ctype = out["selected_contract_type"].astype(str)
    proxy = pd.Series(np.nan, index=out.index, dtype="float64")
    proxy = proxy.mask(ctype.eq("Fix"), out["fixed_volume_mw_std"])
    proxy = proxy.mask(ctype.eq("AsG"), out["generation_mean_mw"])
    proxy = proxy.mask(ctype.eq("AsC"), out["demand_mean_mw"])
    proxy = proxy.mask(ctype.eq("No Contract"), 0.0)
    out["mean_delivered_volume_proxy_mw"] = proxy

    out["mean_delivered_volume_proxy_definition"] = np.select(
        [
            ctype.eq("Fix"),
            ctype.eq("AsG"),
            ctype.eq("AsC"),
            ctype.eq("No Contract"),
        ],
        [
            "optimized fixed volume_mw",
            "scenario mean seller generation MW",
            "scenario mean buyer demand MW",
            "0 MW",
        ],
        default="undefined",
    )

    scenario_name = out.get("scenario_name", pd.Series("", index=out.index)).fillna("").astype(str)
    raw_family = out.get("mutation_family", pd.Series("", index=out.index))
    out["mutation_family_normalized"] = [
        normalize_family_value(fam, scen) for fam, scen in zip(raw_family, scenario_name)
    ]

    out["target_shift_signed"] = out.apply(infer_signed_target_shift, axis=1)
    out["target_shift_abs"] = out["target_shift_signed"].abs()
    out["target_shift_label"] = out["target_shift_signed"].map(_format_shift)

    # Mutation scenario identity. Prefer explicit scenario_name, but include family/target fields to avoid collisions.
    scenario_key_parts = []
    for col in ["scenario_name", "mutation_family_normalized", "target_pair_label", "target_shift_signed", "scenario_order"]:
        if col in out.columns:
            scenario_key_parts.append(col)
    if scenario_key_parts:
        out["_scenario_key"] = out[scenario_key_parts].astype(str).agg(" | ".join, axis=1)
    else:
        out["_scenario_key"] = "scenario"

    out["_scenario_label"] = out.apply(
        lambda r: f"{_short_text(r.get('mutation_family_normalized', 'Mutation'), 18)} | {_format_shift(r.get('target_shift_signed', np.nan))}",
        axis=1,
    )
    out["_stress_order"] = out["target_shift_abs"].fillna(0.0)

    sources = {
        "strike_price_source": strike_source,
        "fixed_volume_source": fixed_volume_source,
        "generation_mean_source": generation_mean_source,
        "demand_mean_source": demand_mean_source,
        "seller_metric_source": seller_metric_source,
        "buyer_metric_source": buyer_metric_source,
        "buyer_slack_source": buyer_slack_source,
    }
    return out, sources


def _active_contract(series: pd.Series) -> pd.Series:
    return ~series.astype(str).isin(["No Contract", "Unknown", "", "nan"])


def _infer_seller_metric_lower_is_better(source: Optional[str]) -> bool:
    if seller_metric_lower_is_better is not None:
        return bool(seller_metric_lower_is_better)
    if source is None:
        return True
    low = str(source).lower()
    if "utility" in low or "revenue" in low:
        return False
    return True


def build_paired_change_data(plot_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, list[str], dict]:
    if "match_id" not in plot_df.columns:
        raise KeyError("Input CSV must contain match_id.")
    if "scenario_type" not in plot_df.columns:
        raise KeyError("Input CSV must contain scenario_type with baseline/mutation labels.")

    df, sources = standardize_result_columns(plot_df)
    diagnostic_cols = find_diagnostic_columns(df)

    scenario_type = df["scenario_type"].fillna("").astype(str).str.lower()
    baseline_mask = scenario_type.eq("baseline")
    baseline = df.loc[baseline_mask].copy()
    mut = df.loc[~baseline_mask].copy()

    if baseline.empty:
        raise ValueError("No baseline rows found in Simulation_Plot_Data_All_Matches.csv.")
    if mut.empty:
        raise ValueError("No mutation rows found in Simulation_Plot_Data_All_Matches.csv.")

    sort_cols = ["match_id"]
    if "scenario_order" in baseline.columns:
        sort_cols.append("scenario_order")
    elif "scenario_name" in baseline.columns:
        sort_cols.append("scenario_name")
    baseline = baseline.sort_values(sort_cols).drop_duplicates("match_id", keep="first")

    baseline_keep = [
        "match_id",
        "selected_contract_type",
        "strike_price_mwh_std",
        "fixed_volume_mw_std",
        "mean_delivered_volume_proxy_mw",
        "mean_delivered_volume_proxy_definition",
        "seller_metric_value",
        "buyer_metric_value",
        "total_metric_value",
        "buyer_participation_slack_value",
        "generation_mean_mw",
        "demand_mean_mw",
    ]
    baseline_keep += [c for c in diagnostic_cols if c not in baseline_keep]
    baseline_keep = [c for c in baseline_keep if c in baseline.columns]

    baseline_prefixed = baseline[baseline_keep].rename(
        columns={c: f"baseline_{c}" for c in baseline_keep if c != "match_id"}
    )

    long = mut.merge(baseline_prefixed, on="match_id", how="left", validate="many_to_one")
    long = long.loc[long["baseline_selected_contract_type"].notna()].copy()

    long["contract_transition"] = (
        long["baseline_selected_contract_type"].astype(str)
        + " → "
        + long["selected_contract_type"].astype(str)
    )
    long["contract_type_changed"] = (
        long["baseline_selected_contract_type"].astype(str) != long["selected_contract_type"].astype(str)
    )

    both_active = _active_contract(long["baseline_selected_contract_type"]) & _active_contract(long["selected_contract_type"])
    long["strike_price_directly_comparable"] = both_active
    long["delta_strike_price_mwh"] = (
        long["strike_price_mwh_std"] - long["baseline_strike_price_mwh_std"]
    ).where(both_active)

    fix_to_fix = long["baseline_selected_contract_type"].astype(str).eq("Fix") & long["selected_contract_type"].astype(str).eq("Fix")
    long["fixed_volume_directly_comparable"] = fix_to_fix
    long["delta_fixed_volume_mw"] = (
        long["fixed_volume_mw_std"] - long["baseline_fixed_volume_mw_std"]
    ).where(fix_to_fix)

    long["delta_mean_delivered_volume_proxy_mw"] = (
        long["mean_delivered_volume_proxy_mw"] - long["baseline_mean_delivered_volume_proxy_mw"]
    )
    long["delta_seller_metric"] = long["seller_metric_value"] - long["baseline_seller_metric_value"]
    long["delta_buyer_metric"] = long["buyer_metric_value"] - long["baseline_buyer_metric_value"]
    long["delta_total_metric"] = long["total_metric_value"] - long["baseline_total_metric_value"]
    long["delta_buyer_participation_slack"] = (
        long["buyer_participation_slack_value"] - long["baseline_buyer_participation_slack_value"]
    )

    # A full-decision-change flag: profile, strike price, and fixed volume when applicable.
    price_same = np.isclose(
        pd.to_numeric(long["strike_price_mwh_std"], errors="coerce"),
        pd.to_numeric(long["baseline_strike_price_mwh_std"], errors="coerce"),
        rtol=1e-9,
        atol=1e-9,
        equal_nan=True,
    )
    fixed_volume_same = np.isclose(
        pd.to_numeric(long["fixed_volume_mw_std"], errors="coerce"),
        pd.to_numeric(long["baseline_fixed_volume_mw_std"], errors="coerce"),
        rtol=1e-9,
        atol=1e-9,
        equal_nan=True,
    )
    nonfix_volume_ok = ~(
        long["baseline_selected_contract_type"].astype(str).eq("Fix") | long["selected_contract_type"].astype(str).eq("Fix")
    )
    long["full_decision_changed"] = ~(
        (~long["contract_type_changed"]) & price_same & (fixed_volume_same | nonfix_volume_ok)
    )

    diagnostic_delta_cols = []
    for col in diagnostic_cols:
        base_col = f"baseline_{col}"
        if col in long.columns and base_col in long.columns:
            safe = re.sub(r"[^A-Za-z0-9_]+", "_", col).strip("_")
            out_col = f"delta_diag_{safe}"
            long[out_col] = pd.to_numeric(long[col], errors="coerce") - pd.to_numeric(long[base_col], errors="coerce")
            diagnostic_delta_cols.append(out_col)

    # Reasonable column order for the saved long table.
    preferred_first = [
        "match_id",
        "scenario_name",
        "scenario_type",
        "_scenario_key",
        "_scenario_label",
        "scenario_order",
        "mutation_family",
        "mutation_family_normalized",
        "target_pair_label",
        "target_shift_signed",
        "target_shift_abs",
        "target_shift_label",
        "baseline_selected_contract_type",
        "selected_contract_type",
        "contract_transition",
        "contract_type_changed",
        "full_decision_changed",
        "baseline_strike_price_mwh_std",
        "strike_price_mwh_std",
        "delta_strike_price_mwh",
        "strike_price_directly_comparable",
        "baseline_fixed_volume_mw_std",
        "fixed_volume_mw_std",
        "delta_fixed_volume_mw",
        "fixed_volume_directly_comparable",
        "baseline_mean_delivered_volume_proxy_mw",
        "mean_delivered_volume_proxy_mw",
        "delta_mean_delivered_volume_proxy_mw",
        "baseline_mean_delivered_volume_proxy_definition",
        "mean_delivered_volume_proxy_definition",
        "baseline_seller_metric_value",
        "seller_metric_value",
        "delta_seller_metric",
        "baseline_buyer_metric_value",
        "buyer_metric_value",
        "delta_buyer_metric",
        "baseline_total_metric_value",
        "total_metric_value",
        "delta_total_metric",
        "baseline_buyer_participation_slack_value",
        "buyer_participation_slack_value",
        "delta_buyer_participation_slack",
    ]
    preferred_first = [c for c in preferred_first if c in long.columns]
    remaining = [c for c in long.columns if c not in preferred_first]
    long = long[preferred_first + remaining].copy()

    sources["diagnostic_columns"] = diagnostic_cols
    sources["diagnostic_delta_columns"] = diagnostic_delta_cols
    sources["seller_metric_lower_is_better_inferred"] = _infer_seller_metric_lower_is_better(sources.get("seller_metric_source"))

    return baseline, long, diagnostic_delta_cols, sources


def apply_analysis_filters(long_df: pd.DataFrame) -> pd.DataFrame:
    out = long_df.copy()

    if filter_match_ids is not None:
        out = out.loc[out["match_id"].isin([int(x) for x in filter_match_ids])].copy()

    if filter_mutation_families is not None:
        keep_raw = {str(x).lower() for x in filter_mutation_families}
        keep_norm = {normalize_family_value(x).lower() for x in filter_mutation_families}
        out = out.loc[
            out["mutation_family_normalized"].astype(str).str.lower().isin(keep_norm)
            | out.get("mutation_family", pd.Series("", index=out.index)).astype(str).str.lower().isin(keep_raw)
        ].copy()

    if filter_target_pair_labels is not None and "target_pair_label" in out.columns:
        keep = {str(x) for x in filter_target_pair_labels}
        out = out.loc[out["target_pair_label"].astype(str).isin(keep)].copy()

    if filter_scenario_names is not None and "scenario_name" in out.columns:
        keep = {str(x) for x in filter_scenario_names}
        out = out.loc[out["scenario_name"].astype(str).isin(keep)].copy()

    if filter_target_shifts is not None:
        keep = np.array([float(x) for x in filter_target_shifts])
        values = pd.to_numeric(out["target_shift_signed"], errors="coerce").to_numpy(dtype=float)
        mask = np.zeros(len(out), dtype=bool)
        for k in keep:
            mask |= np.isclose(values, k, rtol=1e-6, atol=1e-6)
        out = out.loc[mask].copy()

    if out.empty:
        raise ValueError("All rows were removed by filters. Relax the filter controls and rerun.")
    return out


In [ ]:
# ============================================================
# JOURNAL TABLE BUILDERS
# ============================================================


def _q(s, q):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty:
        return np.nan
    return float(s.quantile(q))


def _mean_abs(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty:
        return np.nan
    return float(s.abs().mean())


def _share_positive(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty:
        return np.nan
    return float((s > 0).mean())


def _share_negative(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty:
        return np.nan
    return float((s < 0).mean())


def _ordered_families(df: pd.DataFrame) -> list[str]:
    present = df["mutation_family_normalized"].dropna().astype(str).unique().tolist()
    ordered = [x for x in preferred_family_order if x in present]
    ordered += sorted([x for x in present if x not in ordered])
    return ordered


def _scenario_metadata(long_df: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "_scenario_key",
        "_scenario_label",
        "scenario_name",
        "scenario_order",
        "mutation_family",
        "mutation_family_normalized",
        "target_pair_label",
        "target_shift_signed",
        "target_shift_abs",
        "target_shift_label",
        "_stress_order",
    ]
    cols = [c for c in cols if c in long_df.columns]
    meta = long_df[cols].drop_duplicates("_scenario_key").copy()
    meta["family_order"] = meta["mutation_family_normalized"].map(
        {fam: i for i, fam in enumerate(_ordered_families(long_df))}
    ).fillna(999)
    meta = meta.sort_values(["family_order", "target_shift_abs", "target_shift_signed", "_scenario_key"], na_position="last")
    meta["scenario_plot_order"] = np.arange(len(meta))
    return meta.drop(columns=["family_order"])


def metric_distribution_long(df: pd.DataFrame, group_cols: list[str], metrics: list[str]) -> pd.DataFrame:
    group_cols = [c for c in group_cols if c in df.columns]
    if not group_cols:
        raise ValueError("metric_distribution_long requires at least one valid grouping column.")

    rows = []
    grouped = df.groupby(group_cols, dropna=False)
    for key, sub in grouped:
        if not isinstance(key, tuple):
            key = (key,)
        base = dict(zip(group_cols, key))
        for metric in [m for m in metrics if m in sub.columns]:
            vals = pd.to_numeric(sub[metric], errors="coerce").dropna()
            row = base.copy()
            row.update(
                {
                    "metric": metric,
                    "metric_label": metric_display_names.get(metric, metric),
                    "n_valid": int(vals.size),
                    "mean": float(vals.mean()) if vals.size else np.nan,
                    "median": float(vals.median()) if vals.size else np.nan,
                    "p10": _q(vals, 0.10),
                    "p25": _q(vals, 0.25),
                    "p75": _q(vals, 0.75),
                    "p90": _q(vals, 0.90),
                    "std": float(vals.std(ddof=1)) if vals.size > 1 else np.nan,
                    "min": float(vals.min()) if vals.size else np.nan,
                    "max": float(vals.max()) if vals.size else np.nan,
                    "mean_abs": float(vals.abs().mean()) if vals.size else np.nan,
                    "positive_share": float((vals > 0).mean()) if vals.size else np.nan,
                    "negative_share": float((vals < 0).mean()) if vals.size else np.nan,
                }
            )
            rows.append(row)
    return pd.DataFrame(rows)


def profile_share_tables(long_df: pd.DataFrame, baseline_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    scenario_cols = [
        "_scenario_key",
        "_scenario_label",
        "mutation_family_normalized",
        "target_shift_signed",
        "target_shift_abs",
        "target_shift_label",
        "_stress_order",
    ]
    mut_counts = (
        long_df.groupby(scenario_cols + ["selected_contract_type"], dropna=False)
        .size()
        .rename("count")
        .reset_index()
    )
    denom = mut_counts.groupby("_scenario_key", dropna=False)["count"].transform("sum")
    mut_counts["share"] = mut_counts["count"] / denom
    mut_counts["is_baseline_reference"] = False

    # Baseline counts once, and baseline repeated per family for plotting.
    base_counts = (
        baseline_df.groupby("selected_contract_type", dropna=False)
        .size()
        .rename("count")
        .reset_index()
    )
    base_counts["share"] = base_counts["count"] / base_counts["count"].sum()
    base_counts["_scenario_key"] = "Baseline"
    base_counts["_scenario_label"] = "Baseline"
    base_counts["mutation_family_normalized"] = "Baseline"
    base_counts["target_shift_signed"] = 0.0
    base_counts["target_shift_abs"] = 0.0
    base_counts["target_shift_label"] = "Baseline"
    base_counts["_stress_order"] = 0.0
    base_counts["is_baseline_reference"] = True
    base_once = base_counts[mut_counts.columns]

    families = _ordered_families(long_df)
    base_repeated = []
    for fam in families:
        tmp = base_once.copy()
        tmp["_scenario_key"] = f"Baseline | {fam}"
        tmp["mutation_family_normalized"] = fam
        base_repeated.append(tmp)
    base_repeated = pd.concat(base_repeated, ignore_index=True) if base_repeated else pd.DataFrame(columns=mut_counts.columns)

    profile_for_analysis = pd.concat([base_once, mut_counts], ignore_index=True)
    profile_for_plot = pd.concat([base_repeated, mut_counts], ignore_index=True)

    return profile_for_analysis, profile_for_plot


def mutation_summary_table(long_df: pd.DataFrame) -> pd.DataFrame:
    group_cols = [
        "_scenario_key",
        "_scenario_label",
        "mutation_family_normalized",
        "target_shift_signed",
        "target_shift_abs",
        "target_shift_label",
    ]
    base = (
        long_df.groupby(group_cols, dropna=False)
        .agg(
            n_rows=("match_id", "size"),
            n_matches=("match_id", "nunique"),
            n_contract_type_switches=("contract_type_changed", "sum"),
            contract_type_switch_rate=("contract_type_changed", "mean"),
            n_full_decision_changes=("full_decision_changed", "sum"),
            full_decision_change_rate=("full_decision_changed", "mean"),
            price_delta_defined_share=("delta_strike_price_mwh", lambda s: float(pd.Series(s).notna().mean())),
            fix_to_fix_volume_delta_defined_share=("delta_fixed_volume_mw", lambda s: float(pd.Series(s).notna().mean())),
        )
        .reset_index()
    )

    # Profile counts and shares.
    counts = (
        long_df.groupby(group_cols + ["selected_contract_type"], dropna=False)
        .size()
        .rename("count")
        .reset_index()
    )
    counts["share"] = counts["count"] / counts.groupby(group_cols, dropna=False)["count"].transform("sum")
    for ctype in contract_type_order:
        safe = re.sub(r"[^A-Za-z0-9]+", "_", ctype).strip("_").lower()
        sub = counts.loc[counts["selected_contract_type"].eq(ctype), group_cols + ["count", "share"]].copy()
        sub = sub.rename(columns={"count": f"n_{safe}", "share": f"share_{safe}"})
        base = base.merge(sub, on=group_cols, how="left")

    # Metric medians and interval summaries.
    for metric in [m for m in metrics_to_summarize if m in long_df.columns]:
        stats = (
            long_df.groupby(group_cols, dropna=False)[metric]
            .agg(
                **{
                    f"{metric}_n_valid": "count",
                    f"{metric}_mean": "mean",
                    f"{metric}_median": "median",
                    f"{metric}_p10": lambda s: _q(s, 0.10),
                    f"{metric}_p25": lambda s: _q(s, 0.25),
                    f"{metric}_p75": lambda s: _q(s, 0.75),
                    f"{metric}_p90": lambda s: _q(s, 0.90),
                    f"{metric}_mean_abs": _mean_abs,
                    f"{metric}_positive_share": _share_positive,
                    f"{metric}_negative_share": _share_negative,
                }
            )
            .reset_index()
        )
        base = base.merge(stats, on=group_cols, how="left")

    return base.sort_values(["mutation_family_normalized", "target_shift_abs", "target_shift_signed"])


def transition_summary_table(long_df: pd.DataFrame) -> pd.DataFrame:
    group_cols = [
        "_scenario_key",
        "_scenario_label",
        "mutation_family_normalized",
        "target_shift_signed",
        "target_shift_abs",
        "target_shift_label",
        "baseline_selected_contract_type",
        "selected_contract_type",
        "contract_transition",
    ]
    trans = long_df.groupby(group_cols, dropna=False).size().rename("n").reset_index()
    scenario_total = trans.groupby("_scenario_key", dropna=False)["n"].transform("sum")
    row_total = trans.groupby(["_scenario_key", "baseline_selected_contract_type"], dropna=False)["n"].transform("sum")
    trans["overall_share"] = trans["n"] / scenario_total
    trans["row_share_given_baseline_type"] = trans["n"] / row_total
    return trans.sort_values(["mutation_family_normalized", "target_shift_abs", "baseline_selected_contract_type", "selected_contract_type"])


def severe_transition_summary(long_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for fam, sub in long_df.groupby("mutation_family_normalized", dropna=False):
        target = sub["target_shift_abs"].max()
        sev = sub.loc[np.isclose(sub["target_shift_abs"], target, equal_nan=False)].copy()
        rows.append(sev)
    severe = pd.concat(rows, ignore_index=True) if rows else long_df.iloc[0:0].copy()
    return transition_summary_table(severe)


def objective_slack_quadrant_summary(long_df: pd.DataFrame, sources: dict) -> pd.DataFrame:
    required = ["delta_seller_metric", "delta_buyer_participation_slack"]
    if any(c not in long_df.columns or long_df[c].notna().sum() == 0 for c in required):
        return pd.DataFrame()

    lower_is_better = bool(sources.get("seller_metric_lower_is_better_inferred", True))
    df = long_df.dropna(subset=required).copy()
    df["seller_metric_improved"] = df["delta_seller_metric"] < 0 if lower_is_better else df["delta_seller_metric"] > 0
    df["buyer_slack_improved"] = df["delta_buyer_participation_slack"] > 0
    df["movement_quadrant"] = np.select(
        [
            df["seller_metric_improved"] & df["buyer_slack_improved"],
            df["seller_metric_improved"] & ~df["buyer_slack_improved"],
            ~df["seller_metric_improved"] & df["buyer_slack_improved"],
            ~df["seller_metric_improved"] & ~df["buyer_slack_improved"],
        ],
        [
            "seller improved / buyer slack improved",
            "seller improved / buyer slack worsened",
            "seller worsened / buyer slack improved",
            "seller worsened / buyer slack worsened",
        ],
        default="undefined",
    )

    group_cols = ["_scenario_key", "_scenario_label", "mutation_family_normalized", "target_shift_signed", "target_shift_abs", "target_shift_label"]
    out = df.groupby(group_cols + ["movement_quadrant"], dropna=False).size().rename("n").reset_index()
    out["share"] = out["n"] / out.groupby(group_cols, dropna=False)["n"].transform("sum")
    out["seller_metric_lower_is_better"] = lower_is_better
    return out.sort_values(["mutation_family_normalized", "target_shift_abs", "movement_quadrant"])


def top_movers_table(long_df: pd.DataFrame, metrics: list[str]) -> pd.DataFrame:
    rows = []
    base_cols = [
        "match_id", "_scenario_key", "_scenario_label", "mutation_family_normalized", "target_shift_signed",
        "baseline_selected_contract_type", "selected_contract_type", "contract_transition",
    ]
    base_cols = [c for c in base_cols if c in long_df.columns]
    for metric in [m for m in metrics if m in long_df.columns]:
        sub = long_df[base_cols + [metric]].copy()
        sub = sub.dropna(subset=[metric])
        if sub.empty:
            continue
        sub["metric"] = metric
        sub["metric_label"] = metric_display_names.get(metric, metric)
        sub["metric_value"] = sub[metric]
        sub["metric_abs_value"] = sub[metric].abs()
        sub = sub.sort_values("metric_abs_value", ascending=False).head(int(max_top_movers_per_metric))
        sub = sub.drop(columns=[metric])
        rows.append(sub)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def diagnostic_summary_table(long_df: pd.DataFrame, diagnostic_delta_cols: list[str]) -> pd.DataFrame:
    if not diagnostic_delta_cols:
        return pd.DataFrame()
    group_cols = ["_scenario_key", "_scenario_label", "mutation_family_normalized", "target_shift_signed", "target_shift_abs", "target_shift_label"]
    return metric_distribution_long(long_df, group_cols, diagnostic_delta_cols)


def build_journal_tables(long_df: pd.DataFrame, baseline_df: pd.DataFrame, diagnostic_delta_cols: list[str], sources: dict) -> dict[str, pd.DataFrame]:
    profile_analysis, profile_plot = profile_share_tables(long_df, baseline_df)
    metric_long_by_scenario = metric_distribution_long(
        long_df,
        ["_scenario_key", "_scenario_label", "mutation_family_normalized", "target_shift_signed", "target_shift_abs", "target_shift_label"],
        [m for m in metrics_to_summarize if m in long_df.columns],
    )
    metric_long_by_transition = metric_distribution_long(
        long_df,
        ["contract_transition", "baseline_selected_contract_type", "selected_contract_type"],
        [m for m in metrics_to_summarize if m in long_df.columns],
    )

    source_rows = []
    for key, value in sources.items():
        if isinstance(value, list):
            value_text = "; ".join(value)
        else:
            value_text = str(value)
        source_rows.append({"item": key, "value": value_text})
    source_audit = pd.DataFrame(source_rows)

    tables = {
        "Journal_Long_Change_By_MatchScenario.csv": long_df,
        "Journal_Baseline_By_Match.csv": baseline_df,
        "Journal_Scenario_Metadata.csv": _scenario_metadata(long_df),
        "Journal_Profile_Shares_ByScenario.csv": profile_analysis,
        "Journal_Profile_Shares_ForPlot.csv": profile_plot,
        "Journal_Mutation_Summary_By_FamilyTarget.csv": mutation_summary_table(long_df),
        "Journal_Metric_Distribution_ByScenario_Long.csv": metric_long_by_scenario,
        "Journal_Metric_Distribution_ByTransition_Long.csv": metric_long_by_transition,
        "Journal_Transition_Summary_By_FamilyTarget.csv": transition_summary_table(long_df),
        "Journal_Transition_Summary_Severe_ByFamily.csv": severe_transition_summary(long_df),
        "Journal_Objective_Slack_Quadrants.csv": objective_slack_quadrant_summary(long_df, sources),
        "Journal_Top_Movers_ByMetric.csv": top_movers_table(long_df, metrics_to_summarize),
        "Journal_Diagnostic_Delta_Summary.csv": diagnostic_summary_table(long_df, diagnostic_delta_cols),
        "Journal_Metric_Source_Audit.csv": source_audit,
    }
    return tables


In [ ]:
# ============================================================
# JOURNAL FIGURE HELPERS
# ============================================================


def _metric_label(metric: str) -> str:
    return metric_display_names.get(metric, metric)


def _running_in_notebook() -> bool:
    try:
        shell = get_ipython().__class__.__name__  # noqa: F821
        return shell == "ZMQInteractiveShell"
    except Exception:
        return False


def _print_figure_reading_note(filename_stem: str) -> None:
    if not bool(globals().get("show_figure_reading_notes", True)):
        return
    notes = globals().get("figure_reading_notes", {})
    note = notes.get(filename_stem, "") if isinstance(notes, dict) else ""
    if not note:
        return
    print(f"\nHow to read {filename_stem}:")
    for line in str(note).strip().splitlines():
        line = line.strip()
        if line:
            print(f"  {line}")


def _save_figure(fig: plt.Figure, filename_stem: str) -> list[Path]:
    saved = []
    if save_png:
        p = figure_dir / f"{filename_stem}.png"
        fig.savefig(p, dpi=figure_dpi, bbox_inches="tight")
        saved.append(p)
    if save_pdf:
        p = figure_dir / f"{filename_stem}.pdf"
        fig.savefig(p, bbox_inches="tight")
        saved.append(p)
    if save_svg:
        p = figure_dir / f"{filename_stem}.svg"
        fig.savefig(p, bbox_inches="tight")
        saved.append(p)
    if saved:
        _print_figure_reading_note(filename_stem)
    if display_plots_in_notebook and _running_in_notebook() and saved and saved[0].suffix.lower() == ".png":
        display(Image(filename=str(saved[0])))
    plt.close(fig)
    return saved

def _fig_style(style: Optional[dict], key: str, default=None):
    if isinstance(style, dict):
        return style.get(key, default)
    return default


def _subplot_adjust_from_style(fig: plt.Figure, style: Optional[dict]) -> None:
    if not isinstance(style, dict):
        return
    kwargs = {}
    for key in ["left", "right", "bottom", "top", "wspace", "hspace"]:
        value = style.get(key)
        if value is not None:
            kwargs[key] = value
    if kwargs:
        fig.subplots_adjust(**kwargs)


def _maybe_add_suptitle(fig: plt.Figure, style: Optional[dict], default_title: str = "") -> None:
    if not bool(_fig_style(style, "show_suptitle", False)):
        return
    title = _fig_style(style, "suptitle", default_title)
    if not title:
        return
    fig.suptitle(
        title,
        fontsize=_fig_style(style, "suptitle_fontsize", 14),
        y=_fig_style(style, "suptitle_y", 1.02),
    )


def _apply_axis_text_style(ax, style: Optional[dict], *, tick_fontsize_key: str = "tick_label_fontsize") -> None:
    tick_fontsize = _fig_style(style, tick_fontsize_key, None)
    if tick_fontsize is not None:
        ax.tick_params(axis="both", labelsize=tick_fontsize)


def _scenario_labels_for_family(df: pd.DataFrame, fam: str) -> pd.DataFrame:
    sub = df.loc[df["mutation_family_normalized"].astype(str).eq(fam)].copy()
    cols = ["target_shift_abs", "target_shift_signed", "target_shift_label", "_stress_order"]
    cols = [c for c in cols if c in sub.columns]
    labels = sub[cols].drop_duplicates().sort_values(["target_shift_abs", "target_shift_signed"], na_position="last")
    return labels.reset_index(drop=True)


def _clean_axis_label(text: str, max_len: int = 24) -> str:
    return _short_text(text, max_len=max_len)


def _wrap_label(text: str, max_chars: int = 24) -> str:
    if text is None:
        return ""
    try:
        max_chars = int(max_chars)
    except Exception:
        max_chars = 24
    max_chars = max(max_chars, 8)
    wrapped_lines = []
    for line in str(text).splitlines():
        line = line.strip()
        if not line:
            wrapped_lines.append("")
        elif len(line) <= max_chars:
            wrapped_lines.append(line)
        else:
            wrapped_lines.append(textwrap.fill(line, width=max_chars, break_long_words=False))
    return "\n".join(wrapped_lines)


def _wrap_text_to_rendered_width(text: str, renderer, font_properties, max_width_px: float) -> str:
    """Wrap text so every rendered line fits within max_width_px."""
    paragraphs = str(text).splitlines() or [""]
    wrapped_lines = []

    def rendered_width(value: str) -> float:
        width, _, _ = renderer.get_text_width_height_descent(value, font_properties, ismath=False)
        return float(width)

    for paragraph in paragraphs:
        words = paragraph.split()
        if not words:
            wrapped_lines.append("")
            continue
        current = words[0]
        for word in words[1:]:
            candidate = f"{current} {word}"
            if rendered_width(candidate) <= max_width_px:
                current = candidate
            else:
                wrapped_lines.append(current)
                current = word
        wrapped_lines.append(current)

    return "\n".join(wrapped_lines)


def _wrap_top_row_titles_and_expand_figure(fig: plt.Figure, axes, style: Optional[dict]) -> int:
    """Fit column titles to their axes and add height for wrapped lines."""
    if not bool(_fig_style(style, "wrap_column_titles", False)):
        return 1

    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    width_fraction = float(_fig_style(style, "column_title_width_fraction", 0.90))
    width_fraction = min(max(width_fraction, 0.50), 1.00)
    max_lines = 1

    for ax in np.asarray(axes)[0, :]:
        title_artist = ax.title
        title = title_artist.get_text().strip()
        if not title:
            continue
        axes_width_px = float(ax.get_window_extent(renderer=renderer).width)
        wrapped = _wrap_text_to_rendered_width(
            title, renderer, title_artist.get_fontproperties(), axes_width_px * width_fraction
        )
        title_artist.set_text(wrapped)
        max_lines = max(max_lines, wrapped.count("\n") + 1)

    extra_per_line = float(_fig_style(style, "column_title_extra_height_per_line", 0.35))
    if max_lines > 1 and extra_per_line > 0:
        width, height = fig.get_size_inches()
        fig.set_size_inches(width, height + extra_per_line * (max_lines - 1), forward=True)
        fig.canvas.draw()

    return max_lines


def _contract_terms_metric_label(metric: str, style: Optional[dict] = None) -> str:
    custom_labels = _fig_style(style, "metric_y_labels", {})
    if isinstance(custom_labels, dict) and metric in custom_labels:
        label = custom_labels[metric]
    else:
        label = _metric_label(metric)
    if bool(_fig_style(style, "wrap_y_axis_labels", True)):
        label = _wrap_label(label, _fig_style(style, "y_axis_label_max_chars", 24))
    return label

def _plot_interval_stats(ax, stats: pd.DataFrame, title: str, ylabel: str, style: Optional[dict] = None) -> None:
    if stats.empty:
        ax.axis("off")
        return
    stats = stats.sort_values(["target_shift_abs", "target_shift_signed"], na_position="last").reset_index(drop=True)
    x = np.arange(len(stats))
    med = pd.to_numeric(stats["median"], errors="coerce").to_numpy(dtype=float)
    p10 = pd.to_numeric(stats["p10"], errors="coerce").to_numpy(dtype=float)
    p25 = pd.to_numeric(stats["p25"], errors="coerce").to_numpy(dtype=float)
    p75 = pd.to_numeric(stats["p75"], errors="coerce").to_numpy(dtype=float)
    p90 = pd.to_numeric(stats["p90"], errors="coerce").to_numpy(dtype=float)
    nvalid = pd.to_numeric(stats["n_valid"], errors="coerce").fillna(0).astype(int).to_numpy()
    labels = stats["target_shift_label"].astype(str).tolist()

    n_label_mode = str(_fig_style(style, "n_label_mode", "xtick_second_line"))
    show_n_labels = bool(_fig_style(style, "show_n_labels", True)) and n_label_mode.lower() != "none"
    if show_n_labels and n_label_mode == "xtick_second_line":
        x_tick_labels = [f"{label}\nn={n}" for label, n in zip(labels, nvalid)]
    else:
        x_tick_labels = labels

    ax.axhline(0, linewidth=1)
    ax.vlines(
        x,
        p10,
        p90,
        linewidth=_fig_style(style, "interval_line_width_10_90", 1.1),
        alpha=0.75,
        label="10–90%",
    )
    ax.vlines(
        x,
        p25,
        p75,
        linewidth=_fig_style(style, "interval_line_width_iqr", 4.0),
        alpha=0.9,
        label="IQR",
    )
    plot_kwargs = {
        "marker": "o",
        "linewidth": _fig_style(style, "median_line_width", 1.5),
        "label": "median",
    }
    marker_size = _fig_style(style, "median_marker_size", None)
    if marker_size is not None:
        plot_kwargs["markersize"] = marker_size
    ax.plot(x, med, **plot_kwargs)

    ax.set_xticks(x)
    ax.set_xticklabels(
        x_tick_labels,
        rotation=_fig_style(style, "x_tick_rotation", 0),
        ha=_fig_style(style, "x_tick_ha", "center"),
        fontsize=_fig_style(style, "tick_label_fontsize", None),
    )
    ax.set_title(
        title,
        fontsize=_fig_style(style, "panel_title_fontsize", None),
        pad=_fig_style(style, "panel_title_pad", None),
        wrap=True,
    )
    ax.set_ylabel(
        ylabel,
        fontsize=_fig_style(style, "axis_label_fontsize", None),
        labelpad=_fig_style(style, "y_label_pad", None),
    )
    _apply_axis_text_style(ax, style)

    finite_y = np.concatenate([p10[np.isfinite(p10)], p90[np.isfinite(p90)], med[np.isfinite(med)]])
    if finite_y.size:
        y_min, y_max = float(np.nanmin(finite_y)), float(np.nanmax(finite_y))
        span = y_max - y_min if y_max > y_min else max(abs(y_max), abs(y_min), 1.0)
        ax.set_ylim(y_min - 0.15 * span, y_max + 0.15 * span)

        if show_n_labels and n_label_mode != "xtick_second_line":
            for xi, n in zip(x, nvalid):
                if n_label_mode == "inside_bottom":
                    y_text = y_min + float(_fig_style(style, "n_label_inside_offset", 0.04)) * span
                    ax.text(
                        xi,
                        y_text,
                        f"n={n}",
                        ha="center",
                        va="bottom",
                        fontsize=_fig_style(style, "n_label_fontsize", 7),
                        rotation=_fig_style(style, "n_label_rotation", 0),
                    )
                elif n_label_mode == "below_axis":
                    ax.text(
                        xi,
                        _fig_style(style, "n_label_y_axes", -0.20),
                        f"n={n}",
                        ha="center",
                        va="top",
                        fontsize=_fig_style(style, "n_label_fontsize", 7),
                        rotation=_fig_style(style, "n_label_rotation", 0),
                        transform=ax.get_xaxis_transform(),
                        clip_on=False,
                    )


def _metric_stats_by_family_target(long_df: pd.DataFrame, metric: str) -> pd.DataFrame:
    if metric not in long_df.columns:
        return pd.DataFrame()
    rows = []
    group_cols = ["mutation_family_normalized", "target_shift_signed", "target_shift_abs", "target_shift_label"]
    for key, sub in long_df.groupby(group_cols, dropna=False):
        vals = pd.to_numeric(sub[metric], errors="coerce").dropna()
        row = dict(zip(group_cols, key))
        row.update(
            {
                "n_valid": int(vals.size),
                "median": float(vals.median()) if vals.size else np.nan,
                "p10": _q(vals, 0.10),
                "p25": _q(vals, 0.25),
                "p75": _q(vals, 0.75),
                "p90": _q(vals, 0.90),
            }
        )
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
def plot_profile_share_by_family(profile_for_plot: pd.DataFrame) -> None:
    style = profile_share_figure_style
    families = [fam for fam in preferred_family_order if fam in profile_for_plot["mutation_family_normalized"].unique()]
    families += sorted([fam for fam in profile_for_plot["mutation_family_normalized"].unique() if fam not in families])
    families = [fam for fam in families if fam != "Baseline"]
    if not families:
        return

    ncols = int(_fig_style(style, "ncols", 2))
    nrows = int(math.ceil(len(families) / ncols))
    figsize = (
        float(_fig_style(style, "fig_width", 14.0)),
        max(float(_fig_style(style, "row_height", 4.8)) * nrows, float(_fig_style(style, "min_height", 5.2))),
    )
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, dpi=figure_dpi, squeeze=False)
    axes_flat = axes.ravel()

    for ax, fam in zip(axes_flat, families):
        sub = profile_for_plot.loc[profile_for_plot["mutation_family_normalized"].eq(fam)].copy()
        if sub.empty:
            ax.axis("off")
            continue

        xmeta = (
            sub[["target_shift_abs", "target_shift_signed", "target_shift_label"]]
            .drop_duplicates()
            .sort_values(["target_shift_abs", "target_shift_signed"], na_position="last")
            .reset_index(drop=True)
        )
        xmeta["x"] = np.arange(len(xmeta))
        sub = sub.merge(xmeta, on=["target_shift_abs", "target_shift_signed", "target_shift_label"], how="left")

        for ctype in contract_type_order:
            ct = sub.loc[sub["selected_contract_type"].eq(ctype)].copy()
            if ct.empty:
                continue
            y = ct.groupby("x", dropna=False)["share"].sum().reindex(xmeta["x"]).fillna(0.0).to_numpy(dtype=float)
            if np.nanmax(y) <= 0 and ctype == "Unknown":
                continue
            ax.plot(
                xmeta["x"], y, marker="o", linewidth=_fig_style(style, "line_width", 1.8),
                markersize=_fig_style(style, "marker_size", 5), label=ctype,
            )

        ax.set_title(fam, fontsize=_fig_style(style, "panel_title_fontsize", None), pad=_fig_style(style, "panel_title_pad", None))
        ax.set_ylim(0, 1)
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        ax.set_xticks(xmeta["x"])
        x_tick_labels = ["Reference" if label == "Baseline" else label for label in xmeta["target_shift_label"].astype(str)]
        ax.set_xticklabels(
            x_tick_labels, rotation=_fig_style(style, "x_tick_rotation", 0),
            ha=_fig_style(style, "x_tick_ha", "center"), fontsize=_fig_style(style, "tick_label_fontsize", None),
        )
        ax.set_xlabel(_fig_style(style, "x_axis_label", "Correlation shift"), fontsize=_fig_style(style, "axis_label_fontsize", None), labelpad=_fig_style(style, "x_label_pad", None))
        ax.set_ylabel(_fig_style(style, "y_axis_label", "Selected-profile share"), fontsize=_fig_style(style, "axis_label_fontsize", None), labelpad=_fig_style(style, "y_label_pad", None))
        _apply_axis_text_style(ax, style)
        if bool(_fig_style(style, "show_legend", True)):
            ax.legend(loc=_fig_style(style, "legend_loc", "best"), fontsize=_fig_style(style, "legend_fontsize", 8), frameon=bool(_fig_style(style, "legend_frameon", True)))

    for ax in axes_flat[len(families):]:
        ax.axis("off")

    _subplot_adjust_from_style(fig, style)
    _maybe_add_suptitle(fig, style, "Selected PPA profile shares by mutation family")
    _save_figure(fig, "fig_profile_share_by_family")


def plot_contract_terms_by_family(long_df: pd.DataFrame) -> None:
    style = contract_terms_figure_style
    families = _ordered_families(long_df)
    metrics = [
        "delta_strike_price_mwh",
        "delta_fixed_volume_mw",
        "delta_mean_delivered_volume_proxy_mw",
    ]
    metrics = [m for m in metrics if m in long_df.columns and pd.to_numeric(long_df[m], errors="coerce").notna().any()]
    if not families or not metrics:
        return

    nrows = len(metrics)
    ncols = len(families)
    figsize = (
        max(float(_fig_style(style, "fig_width_per_column", 4.4)) * ncols, float(_fig_style(style, "min_width", 8.0))),
        max(float(_fig_style(style, "fig_height_per_row", 3.7)) * nrows, float(_fig_style(style, "min_height", 6.0))),
    )
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, dpi=figure_dpi, squeeze=False)

    for r, metric in enumerate(metrics):
        stats = _metric_stats_by_family_target(long_df, metric)
        for c, fam in enumerate(families):
            ax = axes[r, c]
            sub = stats.loc[stats["mutation_family_normalized"].astype(str).eq(fam)].copy()
            title = fam if (r == 0 and bool(_fig_style(style, "show_column_titles", True))) else ""
            ylabel = _contract_terms_metric_label(metric, style) if c == 0 else ""
            _plot_interval_stats(ax, sub, title, ylabel, style=style)
            if r == nrows - 1:
                ax.set_xlabel(
                    _fig_style(style, "x_axis_label", "Correlation shift"),
                    fontsize=_fig_style(style, "axis_label_fontsize", None),
                    labelpad=_fig_style(style, "x_label_pad", None),
                )
            if c != 0:
                ax.set_ylabel("")
            if r == 0 and c == ncols - 1 and bool(_fig_style(style, "show_legend", True)):
                ax.legend(
                    loc=_fig_style(style, "legend_loc", "best"),
                    fontsize=_fig_style(style, "legend_fontsize", 8),
                    frameon=bool(_fig_style(style, "legend_frameon", True)),
                )

    _subplot_adjust_from_style(fig, style)
    _wrap_top_row_titles_and_expand_figure(fig, axes, style)
    _maybe_add_suptitle(fig, style, "Contract-term changes under mutation\nMedian with IQR and 10–90% ranges across matches")
    _save_figure(fig, "fig_contract_terms_by_family")

def plot_transition_matrices_by_family_severe(long_df: pd.DataFrame) -> None:
    style = transition_matrix_figure_style
    families = _ordered_families(long_df)
    if not families:
        return

    ncols = int(_fig_style(style, "ncols", 2))
    nrows = int(math.ceil(len(families) / ncols))
    figsize = (
        float(_fig_style(style, "fig_width", 13.5)),
        max(float(_fig_style(style, "row_height", 5.1)) * nrows, float(_fig_style(style, "min_height", 5.5))),
    )
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, dpi=figure_dpi, squeeze=False)
    axes_flat = axes.ravel()
    im = None

    for ax, fam in zip(axes_flat, families):
        sub_f = long_df.loc[long_df["mutation_family_normalized"].astype(str).eq(fam)].copy()
        if sub_f.empty:
            ax.axis("off")
            continue
        severe_abs = sub_f["target_shift_abs"].max()
        sub = sub_f.loc[np.isclose(sub_f["target_shift_abs"], severe_abs, equal_nan=False)].copy()
        if sub.empty:
            ax.axis("off")
            continue
        shift_label = sub["target_shift_label"].dropna().astype(str).iloc[0]

        row_types = [x for x in contract_type_order if x in sub["baseline_selected_contract_type"].astype(str).unique()]
        col_types = [x for x in contract_type_order if x in sub["selected_contract_type"].astype(str).unique()]
        matrix = pd.DataFrame(0, index=row_types, columns=col_types, dtype=int)
        counts = (
            sub.groupby(["baseline_selected_contract_type", "selected_contract_type"], dropna=False)
            .size()
            .rename("n")
            .reset_index()
        )
        for _, row in counts.iterrows():
            r = str(row["baseline_selected_contract_type"])
            c = str(row["selected_contract_type"])
            if r in matrix.index and c in matrix.columns:
                matrix.loc[r, c] = int(row["n"])

        row_totals = matrix.sum(axis=1).replace(0, np.nan)
        share_matrix = matrix.astype(float).div(row_totals, axis=0).fillna(0.0)
        im = ax.imshow(
            share_matrix.to_numpy(dtype=float),
            aspect="auto",
            cmap=_fig_style(style, "cmap", "Blues"),
            vmin=0.0,
            vmax=1.0,
        )
        ax.set_title(
            f"{fam}",
            fontsize=_fig_style(style, "panel_title_fontsize", None),
            pad=_fig_style(style, "panel_title_pad", None),
        )
        ax.set_xticks(np.arange(len(col_types)))
        ax.set_xticklabels(col_types, fontsize=_fig_style(style, "tick_label_fontsize", None))
        ax.set_yticks(np.arange(len(row_types)))
        ax.set_yticklabels(row_types, fontsize=_fig_style(style, "tick_label_fontsize", None))

        x_label = _fig_style(style, "x_axis_label", "")
        y_label = _fig_style(style, "y_axis_label", "")
        ax.set_xlabel(x_label, fontsize=_fig_style(style, "axis_label_fontsize", None))
        ax.set_ylabel(y_label, fontsize=_fig_style(style, "axis_label_fontsize", None))
        _apply_axis_text_style(ax, style)

        threshold = float(_fig_style(style, "text_color_threshold", 0.55))
        dark_text = _fig_style(style, "cell_annotation_color_dark", "black")
        light_text = _fig_style(style, "cell_annotation_color_light", "white")
        for i, r in enumerate(row_types):
            for j, c in enumerate(col_types):
                n = int(matrix.loc[r, c])
                share = float(share_matrix.loc[r, c])
                if n > 0:
                    ax.text(
                        j,
                        i,
                        f"{n}\n{share:.0%}",
                        ha="center",
                        va="center",
                        fontsize=_fig_style(style, "cell_annotation_fontsize", 9),
                        fontweight=_fig_style(style, "cell_annotation_fontweight", "normal"),
                        color=light_text if share >= threshold else dark_text,
                    )

    for ax in axes_flat[len(families):]:
        ax.axis("off")

    _subplot_adjust_from_style(fig, style)
    if bool(_fig_style(style, "show_shared_axis_note", True)):
        fig.text(
            0.5,
            float(_fig_style(style, "shared_axis_note_y", 0.025)),
            _fig_style(style, "shared_axis_note", "Rows = baseline result; columns = dynamic mutation result."),
            ha="center",
            va="center",
            fontsize=_fig_style(style, "shared_axis_note_fontsize", 9),
        )
    if im is not None:
        cbar = fig.colorbar(
            im,
            ax=axes_flat[: len(families)],
            fraction=_fig_style(style, "colorbar_fraction", 0.035),
            pad=_fig_style(style, "colorbar_pad", 0.02),
            shrink=_fig_style(style, "colorbar_shrink", 0.90),
        )
        cbar.set_label(_fig_style(style, "colorbar_label"), fontsize=_fig_style(style, "colorbar_label_fontsize", None))
        cbar.ax.tick_params(labelsize=_fig_style(style, "colorbar_tick_fontsize", None))
        cbar.ax.yaxis.set_major_formatter(PercentFormatter(1.0))

    _maybe_add_suptitle(fig, style, "Baseline-to-mutation transition matrices at the most severe target in each family")
    _save_figure(fig, "fig_transition_matrices_by_family_severe")

def plot_objective_slack_movement(long_df: pd.DataFrame, sources: dict) -> None:

    style = objective_slack_figure_style

    required = ["delta_seller_metric", "delta_buyer_participation_slack"]

    if any(c not in long_df.columns or pd.to_numeric(long_df[c], errors="coerce").notna().sum() == 0 for c in required):

        print("Skipping objective/slack figure: required seller metric or buyer slack column is missing.")

        return



    df = long_df.dropna(subset=required).copy()

    if df.empty:

        return



    families = _ordered_families(df)

    ncols = int(_fig_style(style, "ncols", 2))

    nrows = int(math.ceil(len(families) / ncols))

    figsize = (

        float(_fig_style(style, "fig_width", 13.5)),

        max(float(_fig_style(style, "row_height", 4.9)) * nrows, float(_fig_style(style, "min_height", 5.5))),

    )

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, dpi=figure_dpi, squeeze=False)

    axes_flat = axes.ravel()



    lower_is_better = bool(sources.get("seller_metric_lower_is_better_inferred", True))

    source = sources.get("seller_metric_source")

    seller_axis_note = "lower is better" if lower_is_better else "higher is better"

    detailed_xlabel = f"Δ seller metric ({source}; {seller_axis_note})" if source else "Δ seller metric"

    x_axis_label = _fig_style(style, "x_axis_label", "Δ seller metric")

    if x_axis_label is None or str(x_axis_label).strip().lower() == "auto":

        x_axis_label = detailed_xlabel

    y_axis_label = _fig_style(style, "y_axis_label", "Δ buyer participation slack")



    c_all = pd.to_numeric(df["target_shift_abs"], errors="coerce")

    c_finite = c_all[np.isfinite(c_all)]

    if c_finite.empty:

        color_norm = None

    else:

        vmin = float(c_finite.min())

        vmax = float(c_finite.max())

        if not np.isfinite(vmin) or not np.isfinite(vmax):

            color_norm = None

        else:

            if vmax <= vmin:

                vmax = vmin + 1e-9

            color_norm = Normalize(vmin=vmin, vmax=vmax)

    cmap = _fig_style(style, "cmap", "viridis")

    sc = None



    for ax, fam in zip(axes_flat, families):

        sub = df.loc[df["mutation_family_normalized"].astype(str).eq(fam)].copy()

        if sub.empty:

            ax.axis("off")

            continue

        x = pd.to_numeric(sub["delta_seller_metric"], errors="coerce")

        y = pd.to_numeric(sub["delta_buyer_participation_slack"], errors="coerce")

        c = pd.to_numeric(sub["target_shift_abs"], errors="coerce")

        sc = ax.scatter(

            x,

            y,

            c=c,

            s=_fig_style(style, "scatter_size", 14),

            alpha=_fig_style(style, "scatter_alpha", 0.35),

            cmap=cmap,

            norm=color_norm,

        )

        ax.axhline(0, linewidth=1)

        ax.axvline(0, linewidth=1)

        ax.set_title(

            fam,

            fontsize=_fig_style(style, "panel_title_fontsize", None),

            pad=_fig_style(style, "panel_title_pad", None),

        )

        ax.set_xlabel(

            x_axis_label,

            fontsize=_fig_style(style, "axis_label_fontsize", None),

            labelpad=_fig_style(style, "x_label_pad", None),

        )

        ax.set_ylabel(

            y_axis_label,

            fontsize=_fig_style(style, "axis_label_fontsize", None),

            labelpad=_fig_style(style, "y_label_pad", None),

        )

        _apply_axis_text_style(ax, style)



        centroids = (

            sub.groupby(["target_shift_abs", "target_shift_label"], dropna=False)

            .agg(

                x=("delta_seller_metric", "median"),

                y=("delta_buyer_participation_slack", "median"),

                n=("match_id", "size"),

            )

            .reset_index()

            .sort_values("target_shift_abs")

        )

        ax.scatter(

            centroids["x"],

            centroids["y"],

            c=pd.to_numeric(centroids["target_shift_abs"], errors="coerce"),

            s=_fig_style(style, "centroid_size", 70),

            marker="D",

            cmap=cmap,

            norm=color_norm,

            edgecolor="black",

            linewidth=0.8,

        )

        if bool(_fig_style(style, "show_centroid_labels", False)):

            for _, row in centroids.iterrows():

                ax.annotate(

                    str(row["target_shift_label"]),

                    (row["x"], row["y"]),

                    xytext=(5, 5),

                    textcoords="offset points",

                    fontsize=_fig_style(style, "centroid_label_fontsize", 8),

                )



    for ax in axes_flat[len(families):]:

        ax.axis("off")



    _subplot_adjust_from_style(fig, style)

    if sc is not None:

        try:

            cbar_position = _fig_style(style, "colorbar_position", None)

            if cbar_position is not None:

                cax = fig.add_axes(cbar_position)

                cbar = fig.colorbar(sc, cax=cax)

            else:

                cbar = fig.colorbar(

                    sc,

                    ax=axes_flat[: len(families)],

                    shrink=_fig_style(style, "colorbar_shrink", 0.85),

                    fraction=_fig_style(style, "colorbar_fraction", 0.030),

                    pad=_fig_style(style, "colorbar_pad", 0.02),

                )

            cbar.set_label(_fig_style(style, "colorbar_label", "|correlation shift|"), fontsize=_fig_style(style, "colorbar_label_fontsize", None))

            cbar.ax.tick_params(labelsize=_fig_style(style, "colorbar_tick_fontsize", None))

        except Exception:

            pass



    _maybe_add_suptitle(

        fig,

        style,

        "Seller metric and buyer participation-slack movement by mutation family\n"

        "Small points are match-scenario observations; diamonds are target-level medians.",

    )

    _save_figure(fig, "fig_objective_slack_movement")



def plot_match_switch_heatmap_figure(long_df: pd.DataFrame) -> None:

    style = match_switch_heatmap_figure_style

    if long_df.empty:

        return



    scenario_meta = _scenario_metadata(long_df).head(int(max_scenarios_in_switch_heatmap))

    scenario_keys = scenario_meta["_scenario_key"].tolist()

    df = long_df.loc[long_df["_scenario_key"].isin(scenario_keys)].copy()



    rank = (

        df.groupby("match_id", dropna=False)

        .agg(

            switch_count=("contract_type_changed", "sum"),

            full_decision_change_count=("full_decision_changed", "sum"),

            mean_abs_volume_proxy_change=("delta_mean_delivered_volume_proxy_mw", _mean_abs),

            mean_abs_seller_metric_change=("delta_seller_metric", _mean_abs),

        )

        .sort_values(["switch_count", "full_decision_change_count", "mean_abs_seller_metric_change"], ascending=False)

        .head(int(max_matches_in_switch_heatmap))

        .reset_index()

    )

    if rank.empty:

        return



    match_order = rank["match_id"].tolist()

    pivot = df.pivot_table(index="match_id", columns="_scenario_key", values="selected_contract_type", aggfunc="first")

    pivot = pivot.reindex(index=match_order, columns=scenario_keys)



    code_map = {ct: i for i, ct in enumerate(contract_type_order)}

    code = pivot.apply(lambda col: col.map(lambda x: code_map.get(str(x), code_map.get("Unknown", len(code_map)))))



    figsize = (

        max(float(_fig_style(style, "min_width", 8.0)), float(_fig_style(style, "scenario_width", 0.7)) * len(scenario_keys)),

        max(

            float(_fig_style(style, "min_height", 6.0)),

            float(_fig_style(style, "match_height", 0.22)) * len(match_order) + float(_fig_style(style, "height_padding", 2.8)),

        ),

    )

    fig, ax = plt.subplots(figsize=figsize, dpi=figure_dpi)

    im = ax.imshow(code.to_numpy(dtype=float), aspect="auto", cmap="tab10", vmin=-0.5, vmax=len(contract_type_order)-0.5)

    if bool(_fig_style(style, "show_title", True)):

        ax.set_title(

            _fig_style(style, "title", "Selected profile by match and mutation scenario\n(matches ranked by switch activity)"),

            fontsize=_fig_style(style, "title_fontsize", None),

            pad=_fig_style(style, "title_pad", None),

        )

    ax.set_xlabel("Scenario", fontsize=_fig_style(style, "axis_label_fontsize", None))

    ax.set_ylabel("Match ID", fontsize=_fig_style(style, "axis_label_fontsize", None))

    ax.set_xticks(np.arange(len(scenario_keys)))

    ax.set_xticklabels(

        scenario_meta["_scenario_label"].map(lambda x: _clean_axis_label(x, 22)),

        rotation=_fig_style(style, "x_tick_rotation", 45),

        ha="right",

        fontsize=_fig_style(style, "tick_label_fontsize", 8),

    )

    ax.set_yticks(np.arange(len(match_order)))

    ax.set_yticklabels([str(x) for x in match_order], fontsize=_fig_style(style, "y_tick_label_fontsize", 7))



    cbar = fig.colorbar(im, ax=ax, ticks=np.arange(len(contract_type_order)), shrink=0.85)

    cbar.ax.set_yticklabels(contract_type_order)

    cbar.set_label("Selected profile", fontsize=_fig_style(style, "colorbar_label_fontsize", None))

    cbar.ax.tick_params(labelsize=_fig_style(style, "colorbar_tick_fontsize", None))

    _save_figure(fig, "fig_match_switch_heatmap")





def plot_correlation_spillover_heatmap(long_df: pd.DataFrame, diagnostic_delta_cols: list[str]) -> None:

    style = correlation_spillover_heatmap_figure_style

    corr_cols = [

        c for c in diagnostic_delta_cols

        if any(p in c.lower() for p in correlation_heatmap_patterns)

    ]

    if not corr_cols:

        print("Skipping correlation/spillover heatmap: no correlation-style diagnostic delta columns found.")

        return



    group_cols = ["mutation_family_normalized", "target_shift_label", "target_shift_abs", "target_shift_signed"]

    rows = []

    for key, sub in long_df.groupby(group_cols, dropna=False):

        row = dict(zip(group_cols, key))

        row["row_label"] = f"{_short_text(row['mutation_family_normalized'], 20)} | {row['target_shift_label']}"

        for col in corr_cols:

            row[col] = pd.to_numeric(sub[col], errors="coerce").median()

        rows.append(row)

    mat = pd.DataFrame(rows)

    if mat.empty:

        return

    mat = mat.sort_values(["mutation_family_normalized", "target_shift_abs", "target_shift_signed"], na_position="last")

    values = mat[corr_cols].to_numpy(dtype=float)

    if not np.isfinite(values).any():

        return



    vmax = np.nanquantile(np.abs(values[np.isfinite(values)]), 0.95)

    if not np.isfinite(vmax) or vmax <= 0:

        vmax = np.nanmax(np.abs(values[np.isfinite(values)]))

    if not np.isfinite(vmax) or vmax <= 0:

        vmax = 1.0



    figsize = (

        max(float(_fig_style(style, "min_width", 10.0)), float(_fig_style(style, "column_width", 0.7)) * len(corr_cols)),

        max(

            float(_fig_style(style, "min_height", 6.0)),

            float(_fig_style(style, "row_height", 0.35)) * len(mat) + float(_fig_style(style, "height_padding", 2.5)),

        ),

    )

    fig, ax = plt.subplots(figsize=figsize, dpi=figure_dpi)

    im = ax.imshow(values, aspect="auto", cmap="coolwarm", norm=TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax))

    if bool(_fig_style(style, "show_title", True)):

        ax.set_title(

            _fig_style(style, "title", "Median change in correlation diagnostics by mutation scenario"),

            fontsize=_fig_style(style, "title_fontsize", None),

            pad=_fig_style(style, "title_pad", None),

        )

    ax.set_xticks(np.arange(len(corr_cols)))

    ax.set_xticklabels(

        [c.replace("delta_diag_", "") for c in corr_cols],

        rotation=_fig_style(style, "x_tick_rotation", 45),

        ha="right",

        fontsize=_fig_style(style, "tick_label_fontsize", None),

    )

    ax.set_yticks(np.arange(len(mat)))

    ax.set_yticklabels(mat["row_label"].astype(str).tolist(), fontsize=_fig_style(style, "tick_label_fontsize", None))



    for i in range(values.shape[0]):

        for j in range(values.shape[1]):

            val = values[i, j]

            if np.isfinite(val):

                ax.text(

                    j,

                    i,

                    f"{val:+.3f}",

                    ha="center",

                    va="center",

                    fontsize=_fig_style(style, "cell_annotation_fontsize", 7),

                )



    cbar = fig.colorbar(im, ax=ax, shrink=0.85, label=_fig_style(style, "colorbar_label", "median baseline-to-mutation delta"))

    cbar.set_label(_fig_style(style, "colorbar_label", "median baseline-to-mutation delta"), fontsize=_fig_style(style, "colorbar_label_fontsize", None))

    cbar.ax.tick_params(labelsize=_fig_style(style, "colorbar_tick_fontsize", None))

    _save_figure(fig, "fig_correlation_spillover_heatmap")





def make_all_journal_figures(long_df: pd.DataFrame, tables: dict[str, pd.DataFrame], diagnostic_delta_cols: list[str], sources: dict) -> None:

    if not make_plots:

        return

    if make_profile_share_figure:

        print("Creating fig_profile_share_by_family ...", flush=True)

        plot_profile_share_by_family(tables["Journal_Profile_Shares_ForPlot.csv"])

    if make_transition_matrix_figure:

        print("Creating fig_transition_matrices_by_family_severe ...", flush=True)

        plot_transition_matrices_by_family_severe(long_df)

    if make_contract_terms_figure:

        print("Creating fig_contract_terms_by_family ...", flush=True)

        plot_contract_terms_by_family(long_df)

    if make_objective_slack_figure:

        print("Creating fig_objective_slack_movement ...", flush=True)

        plot_objective_slack_movement(long_df, sources)

    if make_match_switch_heatmap:

        print("Creating fig_match_switch_heatmap ...", flush=True)

        plot_match_switch_heatmap_figure(long_df)

    if make_correlation_spillover_heatmap:

        print("Creating fig_correlation_spillover_heatmap ...", flush=True)

        plot_correlation_spillover_heatmap(long_df, diagnostic_delta_cols)



In [ ]:
# ============================================================
# RUN ANALYSIS
# ============================================================

plot_df_raw = read_csv_optimized(input_csv)
baseline_df, long_df, diagnostic_delta_cols, sources = build_paired_change_data(plot_df_raw)
long_df = apply_analysis_filters(long_df)

# Refit baseline universe to selected matches after filters.
baseline_df_filtered = baseline_df.loc[baseline_df["match_id"].isin(long_df["match_id"].unique())].copy()

journal_tables = build_journal_tables(long_df, baseline_df_filtered, diagnostic_delta_cols, sources)

for filename, table in journal_tables.items():
    table.to_csv(table_dir / filename, index=False, float_format="%.8g")

make_all_journal_figures(long_df, journal_tables, diagnostic_delta_cols, sources)

print("Saved journal-ready contract-change outputs to:")
print(f"  Tables:  {table_dir}")
print(f"  Figures: {figure_dir}")

print("\nSource audit:")
for key, value in sources.items():
    if isinstance(value, list):
        print(f"  {key}: {len(value)} column(s)")
    else:
        print(f"  {key}: {value}")

print("\nMain table files:")
for filename in journal_tables:
    print(f"  - {filename}")

if make_plots:
    print("\nMain figure files:")
    for p in sorted(figure_dir.glob("fig_*")):
        print(f"  - {p.name}")

print("\nKey counts:")
print("  Paired mutation rows:", len(long_df))
print("  Unique matches:", long_df["match_id"].nunique())
print("  Mutation scenarios:", long_df["_scenario_key"].nunique())
print("  Contract-type switch count:", int(long_df["contract_type_changed"].sum()))
print("  Contract-type switch rate:", round(float(long_df["contract_type_changed"].mean()), 4))

summary_cols = [
    "_scenario_label", "n_matches", "n_contract_type_switches", "contract_type_switch_rate",
    "share_fix", "share_asc", "share_asg",
    "delta_strike_price_mwh_median",
    "delta_fixed_volume_mw_median",
    "delta_mean_delivered_volume_proxy_mw_median",
    "delta_seller_metric_median",
    "delta_buyer_participation_slack_median",
]
summary_cols = [c for c in summary_cols if c in journal_tables["Journal_Mutation_Summary_By_FamilyTarget.csv"].columns]
print("\nMutation summary preview:")
display(journal_tables["Journal_Mutation_Summary_By_FamilyTarget.csv"][summary_cols].head(20))


In [ ]:
# ============================================================
# OPTIONAL PREVIEW TABLES
# ============================================================

if show_preview_tables:
    print("Scenario metadata")
    display(journal_tables["Journal_Scenario_Metadata.csv"].head(20))

    print("Profile shares for plotting")
    display(journal_tables["Journal_Profile_Shares_ForPlot.csv"].head(20))

    print("Transition summary")
    display(journal_tables["Journal_Transition_Summary_By_FamilyTarget.csv"].head(20))

    print("Metric distribution summary")
    display(journal_tables["Journal_Metric_Distribution_ByScenario_Long.csv"].head(20))

    if not journal_tables["Journal_Objective_Slack_Quadrants.csv"].empty:
        print("Seller metric / buyer slack movement quadrants")
        display(journal_tables["Journal_Objective_Slack_Quadrants.csv"].head(20))

    if not journal_tables["Journal_Diagnostic_Delta_Summary.csv"].empty:
        print("Diagnostic delta summary")
        display(journal_tables["Journal_Diagnostic_Delta_Summary.csv"].head(20))
